In [4]:

%%writefile app/main_basic.py
"""
최소한의 FastAPI 서버
"""
from fastapi import FastAPI

# FastAPI 인스턴스 생성
app = FastAPI(
    title="My First ML API",
    description="Day 2 실습: 첫 번째 FastAPI 서버",
    version="0.1.0",
)

# 엔드포인트 1: 헬스체크 (서버가 살아있는지 확인)
@app.get("/health")
def health_check():
    return {"status": "healthy"}

# 엔드포인트 2: 루트 경로
@app.get("/")
def root():
    return {
        "message": "ML Model Serving API",
        "docs_url": "/docs",
    }

Writing app/main_basic.py


In [6]:
# 노트북 환경에서 uvicorn을 실행하기 위한 설정
!pip install nest_asyncio -q

In [7]:
import nest_asyncio
import uvicorn
import threading
import asyncio

nest_asyncio.apply()

def run_server():
    # 1. 현재 스레드(Thread-5)를 위한 새로운 이벤트 루프 생성 및 설정
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    
    # 2. 서버 실행
    # 주의: app.main_basic:app 파일이 실제로 존재해야 합니다!
    config = uvicorn.Config("app.main_basic:app", host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

# 기존 서버가 혹시 떠 있다면 종료하기 어렵기 때문에 포트 번호를 바꾸거나 
# 커널을 재시작하고 실행하는 것을 추천합니다.
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

import time
time.sleep(2) 
print("✅ 서버 확인 중: http://localhost:8000")

INFO:     Started server process [20698]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ (진짜로) 서버 확인 중: http://localhost:8000
INFO:     127.0.0.1:57022 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:57022 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:57034 - "GET / HTTP/1.1" 200 OK


In [8]:
import requests

# 헬스체크 엔드포인트 호출
response = requests.get("http://localhost:8000/health")
print(f"상태 코드: {response.status_code}")
print(f"응답: {response.json()}")


INFO:     127.0.0.1:52830 - "GET /health HTTP/1.1" 200 OK
상태 코드: 200
응답: {'status': 'healthy'}
INFO:     127.0.0.1:57920 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:37998 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:37998 - "GET /favicon.ico HTTP/1.1" 404 Not Found


In [9]:
# 루트 엔드포인트 호출
response = requests.get("http://localhost:8000/")
print(f"상태 코드: {response.status_code}")
print(f"응답: {response.json()}")

INFO:     127.0.0.1:49638 - "GET / HTTP/1.1" 200 OK
상태 코드: 200
응답: {'message': 'ML Model Serving API', 'docs_url': '/docs'}


# 엔드포인트 만들기 연습

## Path 파라미터

In [10]:

%%writefile app/main_params.py
"""
파라미터 방식 실습
"""
from fastapi import FastAPI

app = FastAPI(title="Parameter Examples")

# ===== Path 파라미터 =====

# 기본 사용: 중괄호 {}로 경로 변수를 선언합니다
@app.get("/models/{model_name}")
def get_model_info(model_name: str):
    """특정 모델의 정보를 반환합니다."""
    return {
        "model_name": model_name,
        "status": "running",
        "version": "1.0.0",
    }

Writing app/main_params.py


In [ ]:

# 서버 실행 (이전 서버가 실행 중이면 커널 재시작 후 실행)
import nest_asyncio, uvicorn, threading, time
nest_asyncio.apply()

def run_server():
    uvicorn.run("app.main_params:app", host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)
print("✅ 서버 시작됨")

In [1]:
import nest_asyncio
import uvicorn
import threading
import asyncio

nest_asyncio.apply()

def run_server():
    # 1. 현재 스레드(Thread-5)를 위한 새로운 이벤트 루프 생성 및 설정
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    
    # 2. 서버 실행
    # 주의: app.main_basic:app 파일이 실제로 존재해야 합니다!
    config = uvicorn.Config("app.main_params:app", host="0.0.0.0", port=8000, reload=True, log_level="info")
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

# 기존 서버가 혹시 떠 있다면 종료하기 어렵기 때문에 포트 번호를 바꾸거나 
# 커널을 재시작하고 실행하는 것을 추천합니다.
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

import time
time.sleep(2) 
print("✅ 서버 확인 중: http://localhost:8000")

INFO:     Will watch for changes in these directories: ['/home/onegem/work/model-serving-course']
INFO:     Started server process [32130]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ 서버 확인 중: http://localhost:8000


In [3]:
import requests

# Path 파라미터 테스트
response = requests.get("http://localhost:8000/models/sentiment-v1")
print(response.json())
# {'model_name': 'sentiment-v1', 'status': 'running', 'version': '1.0.0'}

response = requests.get("http://localhost:8000/models/image-classifier")
print(response.json())
# {'model_name': 'image-classifier', 'status': 'running', 'version': '1.0.0'}

INFO:     127.0.0.1:51124 - "GET /models/sentiment-v1 HTTP/1.1" 200 OK
{'model_name': 'sentiment-v1', 'status': 'running', 'version': '1.0.0'}
INFO:     127.0.0.1:51136 - "GET /models/image-classifier HTTP/1.1" 200 OK
{'model_name': 'image-classifier', 'status': 'running', 'version': '1.0.0'}
INFO:     127.0.0.1:49840 - "GET /models/sentiment-v1 HTTP/1.1" 200 OK
INFO:     127.0.0.1:49844 - "GET /models/image-classifier HTTP/1.1" 200 OK
INFO:     127.0.0.1:49852 - "GET /models/qgrrqgqrgrqg HTTP/1.1" 200 OK
INFO:     127.0.0.1:35406 - "GET /models/%E3%85%82%E3%85%88%E3%84%B1%E3%85%82%E3%85%8E%E3%85%82 HTTP/1.1" 200 OK


## 타입 지정 효과 만들기

In [5]:
%%writefile -a app/main_params.py

# Path 파라미터에 int 타입 지정
@app.get("/predictions/{prediction_id}")
def get_prediction(prediction_id: int):
    """특정 예측 결과를 조회합니다."""
    return {
        "prediction_id": prediction_id,
        "label": "긍정",
        "confidence": 0.92,
    }

Appending to app/main_params.py


### 자동으로 검증기능까지 포함 되었다!

In [3]:
import requests

# 정상 요청: 숫자를 전달
response = requests.get("http://localhost:8000/predictions/42")
print(f"상태: {response.status_code}, 응답: {response.json()}")
# 상태: 200, 응답: {'prediction_id': 42, 'label': '긍정', 'confidence': 0.92}

# 잘못된 요청: 문자열을 전달
response = requests.get("http://localhost:8000/predictions/abc")
print(f"상태: {response.status_code}")
print(f"에러: {response.json()}")
# 상태: 422
# 에러: {'detail': [{'type': 'int_parsing', 'msg': 'Input should be a valid integer...'}]}

INFO:     127.0.0.1:58352 - "GET /predictions/42 HTTP/1.1" 200 OK
상태: 200, 응답: {'prediction_id': 42, 'label': '긍정', 'confidence': 0.92}
INFO:     127.0.0.1:58356 - "GET /predictions/abc HTTP/1.1" 422 Unprocessable Entity
상태: 422
에러: {'detail': [{'type': 'int_parsing', 'loc': ['path', 'prediction_id'], 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'abc'}]}
INFO:     127.0.0.1:40222 - "GET /predictions/42 HTTP/1.1" 200 OK
INFO:     127.0.0.1:40232 - "GET /predictions/asd HTTP/1.1" 422 Unprocessable Entity
INFO:     127.0.0.1:40244 - "GET /.well-known/appspecific/com.chrome.devtools.json HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:40244 - "GET /predictions/asd HTTP/1.1" 422 Unprocessable Entity
INFO:     127.0.0.1:40244 - "GET /.well-known/appspecific/com.chrome.devtools.json HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:55656 - "GET /predictions/42 HTTP/1.1" 200 OK
INFO:     127.0.0.1:55656 - "GET /.well-known/appspecific/com.chrome.devtools.json 

## Query 파라미터

In [4]:
%%writefile -a app/main_params.py

# ===== Query 파라미터 =====

# 함수 인자 중 Path에 포함되지 않은 것은 자동으로 Query 파라미터가 됩니다
@app.get("/models")
def list_models(status: str = None, limit: int = 10):
    """
    모델 목록을 조회합니다.

    - status: 필터링 조건 (선택) — "running", "stopped" 등
    - limit: 반환할 최대 개수 (기본값: 10)
    """
    # 실제로는 DB에서 조회하겠지만, 여기서는 예시 데이터를 반환합니다
    models = [
        {"name": "sentiment-v1", "status": "running"},
        {"name": "image-clf-v2", "status": "running"},
        {"name": "ner-v1", "status": "stopped"},
    ]

    # status 필터링
    if status:
        models = [m for m in models if m["status"] == status]

    # limit 적용
    models = models[:limit]

    return {
        "total": len(models),
        "models": models,
    }

Appending to app/main_params.py


In [3]:
import requests
import json

# 파라미터 없이 호출 (기본값 사용)
response = requests.get("http://localhost:8000/models")
print("전체 모델:", json.dumps(response.json(), indent=4, ensure_ascii=False))

# status로 필터링
response = requests.get("http://localhost:8000/models?status=running")
print("running만:", json.dumps(response.json(), indent=4, ensure_ascii=False))

# 여러 파라미터 조합
response = requests.get("http://localhost:8000/models?status=running&limit=1")
print("running, 1개만:", json.dumps(response.json(), indent=4, ensure_ascii=False))

전체 모델: {
    "total": 3,
    "models": [
        {
            "name": "sentiment-v1",
            "status": "running"
        },
        {
            "name": "image-clf-v2",
            "status": "running"
        },
        {
            "name": "ner-v1",
            "status": "stopped"
        }
    ]
}
running만: {
    "total": 2,
    "models": [
        {
            "name": "sentiment-v1",
            "status": "running"
        },
        {
            "name": "image-clf-v2",
            "status": "running"
        }
    ]
}
running, 1개만: {
    "total": 1,
    "models": [
        {
            "name": "sentiment-v1",
            "status": "running"
        }
    ]
}


## Request Body

In [4]:

%%writefile -a app/main_params.py

# ===== Request Body =====
from pydantic import BaseModel
from typing import Optional

# 입력 스키마 정의
class PredictRequest(BaseModel):
    text: str
    return_probabilities: bool = False    # 선택, 기본값 False

# 출력 스키마 정의
class PredictResponse(BaseModel):
    label: str
    confidence: float
    probabilities: Optional[dict] = None

@app.post("/predict", response_model=PredictResponse)
def predict(request: PredictRequest):
    """
    텍스트 감성 분석을 수행합니다.

    - text: 분석할 텍스트 (필수)
    - return_probabilities: 전체 확률을 반환할지 여부 (선택, 기본 False)
    """
    # 실제로는 모델 추론을 수행하겠지만, 여기서는 더미 결과를 반환합니다
    result = {
        "label": "긍정",
        "confidence": 0.92,
    }

    if request.return_probabilities:
        result["probabilities"] = {
            "긍정": 0.92,
            "부정": 0.05,
            "중립": 0.03,
        }

    return result

Appending to app/main_params.py


In [5]:

# POST 요청: JSON 데이터를 본문에 담아 전송
response = requests.post(
    "http://localhost:8000/predict",
    json={"text": "이 영화 정말 재밌다"}
)
print("기본 응답:", response.json())

# 옵션 추가
response = requests.post(
    "http://localhost:8000/predict",
    json={
        "text": "이 영화 정말 재밌다",
        "return_probabilities": True
    }
)
print("확률 포함:", response.json())

기본 응답: {'label': '긍정', 'confidence': 0.92, 'probabilities': None}
확률 포함: {'label': '긍정', 'confidence': 0.92, 'probabilities': {'긍정': 0.92, '부정': 0.05, '중립': 0.03}}


In [8]:

# text 필드 누락
response = requests.post(
    "http://localhost:8000/predict",
    json={"return_probabilities": True}
)
print(f"상태: {response.status_code}")
print(f"에러: {response.json()['detail'][0]['msg']}")
# 상태: 422
# 에러: Field required

# text에 잘못된 타입 전달
response = requests.post(
    "http://localhost:8000/predict",
    json={"text": 12345}
)
print(f"상태: {response.status_code}")
print(f"에러: {response.json()['detail'][0]['msg']}")
# 상태: 422
# 에러: Input should be a valid string

상태: 422
에러: Field required
상태: 422
에러: Input should be a valid string


## Swagger UI 자동 생성 되는 과정

In [9]:
# FastAPI가 자동 생성한 OpenAPI 스펙 확인
import requests, json

response = requests.get("http://localhost:8000/openapi.json")
spec = response.json()

print(f"API 제목: {spec['info']['title']}")
print(f"API 버전: {spec['info']['version']}")
print(f"\n등록된 엔드포인트:")
for path, methods in spec['paths'].items():
    for method in methods:
        print(f"  {method.upper():6s} {path}")

API 제목: Parameter Examples
API 버전: 0.1.0

등록된 엔드포인트:
  GET    /models/{model_name}
  GET    /predictions/{prediction_id}
  GET    /models
  POST   /predict


In [10]:
# PredictRequest의 JSON Schema 확인
predict_schema = spec['components']['schemas']['PredictRequest']
print("PredictRequest 스키마:")
print(json.dumps(predict_schema, indent=2, ensure_ascii=False))

PredictRequest 스키마:
{
  "properties": {
    "text": {
      "type": "string",
      "title": "Text"
    },
    "return_probabilities": {
      "type": "boolean",
      "title": "Return Probabilities",
      "default": false
    }
  },
  "type": "object",
  "required": [
    "text"
  ],
  "title": "PredictRequest"
}


In [11]:
from pydantic import BaseModel, Field

# Field()에 description과 examples를 추가하면 Swagger UI에 반영됩니다
class PredictRequest(BaseModel):
    text: str = Field(
        ...,
        min_length=1,
        max_length=5000,
        description="분석할 텍스트. 1자 이상 5000자 이하.",
        examples=["이 영화 정말 재밌다"],
    )
    return_probabilities: bool = Field(
        default=False,
        description="True로 설정하면 각 클래스별 확률을 함께 반환합니다.",
    )


In [12]:
from app.model_utils import load_model, predict, preprocess
print("✅ model_utils import 성공")

# 모델 로드 테스트
model = load_model("models/mnist_state_dict.pth")
print(f"✅ 모델 로드 성공: {type(model).__name__}")

✅ model_utils import 성공
✅ 모델 로드 성공: SimpleClassifier


In [15]:
%%writefile app/schemas.py
"""
API 입출력 스키마 정의
"""

from pydantic import BaseModel, Field
from typing import Optional


class PredictRequest(BaseModel):
    """모델 추론 요청 스키마"""
    pixel_values: list[float] = Field(
        ...,
        min_length=784,       # 28 * 28 = 784
        max_length=784,
        description="28x28 이미지의 픽셀 값 (784개). 0.0~1.0 범위.",
        examples=[[0.0] * 784],   # Swagger UI에 예시로 표시
    )
    return_probabilities: bool = Field(
        default=False,
        description="True로 설정하면 전체 클래스별 확률을 함께 반환합니다.",
    )


class PredictResponse(BaseModel):
    """모델 추론 응답 스키마"""
    label: int = Field(
        description="예측된 숫자 (0~9)",
    )
    confidence: float = Field(
        description="예측 확신도 (0.0~1.0)",
    )
    probabilities: Optional[list[float]] = Field(
        default=None,
        description="각 클래스(0~9)별 확률. return_probabilities=True일 때만 포함.",
    )
    model_version: str = Field(
        default="1.0.0",
        description="사용된 모델 버전",
    )


class HealthResponse(BaseModel):
    """헬스체크 응답 스키마"""
    status: str
    model_loaded: bool

Writing app/schemas.py


In [16]:
%%writefile app/main.py
"""
Day 2 실습: 모델 추론 API 서버
"""

from fastapi import FastAPI, HTTPException
import torch

from app.model_utils import load_model, predict
from app.schemas import (
    PredictRequest,
    PredictResponse,
    HealthResponse,
)

# ===== FastAPI 앱 생성 =====
app = FastAPI(
    title="MNIST Prediction API",
    description="Day 2 실습: MNIST 숫자 분류 모델 추론 API",
    version="1.0.0",
)


# ===== 모델을 서버 시작 시 한 번만 로드 =====
# 모듈 레벨에서 로드하면 서버가 시작될 때 실행됩니다.
# 요청마다 로드하면 매번 수 초가 걸리므로, 반드시 한 번만 로드해야 합니다.
try:
    model = load_model("models/mnist_state_dict.pth")
    model_loaded = True
    print("✅ 모델 로드 완료")
except Exception as e:
    model = None
    model_loaded = False
    print(f"❌ 모델 로드 실패: {e}")


# ===== 엔드포인트 1: 헬스체크 =====
@app.get("/health", response_model=HealthResponse)
def health_check():
    """서버 상태와 모델 로드 여부를 확인합니다."""
    return HealthResponse(
        status="healthy",
        model_loaded=model_loaded,
    )


# ===== 엔드포인트 2: 모델 추론 =====
@app.post("/predict", response_model=PredictResponse, summary="MNIST 숫자 예측")
def predict_digit(request: PredictRequest):
    """
    28x28 이미지의 픽셀 값을 받아 숫자(0~9)를 예측합니다.

    - **pixel_values**: 784개의 float 리스트 (28x28 이미지)
    - **return_probabilities**: True로 설정하면 전체 확률 분포를 반환
    """
    # 1. 모델이 로드되었는지 확인
    if not model_loaded:
        raise HTTPException(
            status_code=503,
            detail="모델이 로드되지 않았습니다. 서버 로그를 확인하세요."
        )

    # 2. 입력 데이터를 텐서로 변환
    try:
        input_tensor = torch.tensor(request.pixel_values, dtype=torch.float32)
        input_tensor = input_tensor.reshape(1, 1, 28, 28)  # (batch, channel, H, W)
    except Exception as e:
        raise HTTPException(
            status_code=400,
            detail=f"입력 데이터를 텐서로 변환할 수 없습니다: {str(e)}"
        )

    # 3. 추론 실행
    try:
        result = predict(model, input_tensor)
    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=f"모델 추론 중 에러가 발생했습니다: {str(e)}"
        )

    # 4. 응답 생성
    response = PredictResponse(
        label=result["label"],
        confidence=result["confidence"],
        model_version="1.0.0",
    )

    # 5. 옵션: 확률 분포 포함
    if request.return_probabilities:
        response.probabilities = [round(p, 4) for p in result["probabilities"]]

    return response

Writing app/main.py


In [17]:
import requests

response = requests.get("http://localhost:8000/health")
print(f"상태 코드: {response.status_code}")
print(f"응답: {response.json()}")

상태 코드: 200
응답: {'status': 'healthy', 'model_loaded': True}


In [18]:
from torchvision import datasets, transforms

# MNIST 테스트 데이터 로드
test_dataset = datasets.MNIST(
    root="data", train=False, download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])
)

# 첫 번째 테스트 이미지 가져오기
test_image, true_label = test_dataset[0]
print(f"이미지 크기: {test_image.shape}")     # torch.Size([1, 28, 28])
print(f"정답 레이블: {true_label}")

# 픽셀 값을 리스트로 변환 (API에 보낼 형식)
pixel_values = test_image.flatten().tolist()
print(f"픽셀 값 개수: {len(pixel_values)}")   # 784

이미지 크기: torch.Size([1, 28, 28])
정답 레이블: 7
픽셀 값 개수: 784


In [19]:
import json

# 추론 요청
response = requests.post(
    "http://localhost:8000/predict",
    json={
        "pixel_values": pixel_values,
        "return_probabilities": False,
    }
)

print(f"상태 코드: {response.status_code}")
print(f"응답:")
print(json.dumps(response.json(), indent=2, ensure_ascii=False))



상태 코드: 200
응답:
{
  "label": 7,
  "confidence": 1.0,
  "probabilities": null,
  "model_version": "1.0.0"
}


In [20]:
# return_probabilities를 True로 설정
response = requests.post(
    "http://localhost:8000/predict",
    json={
        "pixel_values": pixel_values,
        "return_probabilities": True,
    }
)

result = response.json()
print(f"예측: {result['label']} (확신도: {result['confidence']})")
print(f"\n클래스별 확률:")
for i, prob in enumerate(result['probabilities']):
    bar = "█" * int(prob * 50)
    print(f"  {i}: {prob:.4f} {bar}")

예측: 7 (확신도: 1.0)

클래스별 확률:
  0: 0.0000 
  1: 0.0000 
  2: 0.0000 
  3: 0.0000 
  4: 0.0000 
  5: 0.0000 
  6: 0.0000 
  7: 1.0000 ██████████████████████████████████████████████████
  8: 0.0000 
  9: 0.0000 


In [21]:
# 10개 이미지를 연속으로 테스트
print(f"{'이미지':<8} {'정답':<6} {'예측':<6} {'확신도':<10} {'결과'}")
print("-" * 45)

correct = 0
for i in range(10):
    image, true_label = test_dataset[i]
    pixel_values = image.flatten().tolist()

    response = requests.post(
        "http://localhost:8000/predict",
        json={"pixel_values": pixel_values}
    )
    result = response.json()

    is_correct = result["label"] == true_label
    if is_correct:
        correct += 1

    mark = "✅" if is_correct else "❌"
    print(f"  #{i:<5} {true_label:<6} {result['label']:<6} {result['confidence']:<10} {mark}")

print(f"\n정확도: {correct}/10 ({correct * 10}%)")

이미지      정답     예측     확신도        결과
---------------------------------------------
  #0     7      7      1.0        ✅
  #1     2      2      1.0        ✅
  #2     1      1      1.0        ✅
  #3     0      0      1.0        ✅
  #4     4      4      1.0        ✅
  #5     1      1      1.0        ✅
  #6     4      4      1.0        ✅
  #7     9      9      0.9985     ✅
  #8     5      5      0.9989     ✅
  #9     9      9      1.0        ✅

정확도: 10/10 (100%)


In [22]:
# 784개가 아닌 100개만 전송
response = requests.post(
    "http://localhost:8000/predict",
    json={"pixel_values": [0.0] * 100}
)
print(f"상태 코드: {response.status_code}")  # 422
print(f"에러 메시지: {response.json()['detail'][0]['msg']}")

상태 코드: 422
에러 메시지: List should have at least 784 items after validation, not 100


In [23]:
# 숫자가 아닌 문자열 전달
response = requests.post(
    "http://localhost:8000/predict",
    json={"pixel_values": "이것은 이미지가 아닙니다"}
)
print(f"상태 코드: {response.status_code}")  # 422

상태 코드: 422


In [24]:
# pixel_values 없이 요청
response = requests.post(
    "http://localhost:8000/predict",
    json={"return_probabilities": True}
)
print(f"상태 코드: {response.status_code}")  # 422
print(f"에러: {response.json()['detail'][0]['msg']}")

상태 코드: 422
에러: Field required


In [25]:
response = requests.post(
    "http://localhost:8000/predict",
    json={}
)
print(f"상태 코드: {response.status_code}")  # 422

상태 코드: 422
